In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -*- coding: utf-8 -*-
import copy
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.stats import norm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def find_project_root():
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parents[3])

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd / "Grad_Research",
        cwd.parent,
        Path("/content/Grad_Research"),
        Path("/content/drive/MyDrive/Grad_Research"),
        Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
    ])

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the Grad_Research project root. "
        "In Colab, place the repository at MyDrive/Grad_Research or /content/Grad_Research."
    )


ROOT        = find_project_root()
EXP021_DIR  = ROOT / "EXP021"
MODEL_DIR   = EXP021_DIR / "models"
RESULTS_DIR = EXP021_DIR / "results"

print(f"Device     : {device}")
print(f"Project root: {ROOT}")
print(f"Model dir  : {MODEL_DIR}")
print(f"Results dir: {RESULTS_DIR}")

In [ ]:
@dataclass
class Config:
    # Network
    # input_size=2: [theta_hat, log(posterior_variance)]
    input_size:    int   = 2
    first_hidden:  int   = 100
    second_hidden: int   = 50
    dropout_rate:  float = 0.0

    # Training
    test_length:         int   = 40
    gamma:               float = 0.1
    memory_capacity:     int   = 1000
    epsilon:             float = 0.1
    batch_size:          int   = 128
    q_network_iteration: int   = 40
    learning_rate:       float = 1e-3
    training_size:       int   = 1000
    validation_size:     int   = 200
    validation_interval: int   = 50

    # EAP quadrature
    n_quad:      int   = 61
    prior_mean:  float = 0.0
    prior_std:   float = 1.0

    # Bank / prior
    bank_type: str = "uncor"   # "uncor" | "cor"
    bank_id:   int = 1
    prior:     str = "normal"  # "normal" | "uniform"
    n_items:   int = 500

In [ ]:
from typing import Tuple


def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    p = (1 - c) / (1 + np.exp(-D * a * (theta - b))) + c
    resp = (np.random.random(size=p.shape) <= p).astype(int)
    return resp


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    info = (
        D**2 * a**2 * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )
    return info


def EAP_quadrature(
    item_paras: np.ndarray,
    resp: np.ndarray,
    n_quad: int = 61,
    prior_mean: float = 0.0,
    prior_std: float = 1.0,
    D: float = 1.0,
) -> Tuple[float, float]:
    """EAP mean and posterior variance via Gauss-Hermite-style quadrature.

    Parameters
    ----------
    item_paras : (n_items, 3) array of [a, b, c]
    resp       : (n_items,)  array of 0/1 responses

    Returns
    -------
    (eap_mean, eap_variance)
    """
    theta_grid = np.linspace(
        prior_mean - 4 * prior_std,
        prior_mean + 4 * prior_std,
        n_quad,
    )
    # log prior (normal)
    log_prior = norm.logpdf(theta_grid, prior_mean, prior_std)

    # log likelihood
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    # shape: (n_items, n_quad)
    p = c[:, None] + (1 - c[:, None]) / (
        1 + np.exp(-D * a[:, None] * (theta_grid[None, :] - b[:, None]))
    )
    p = np.clip(p, 1e-10, 1 - 1e-10)
    log_lik = np.sum(
        resp[:, None] * np.log(p) + (1 - resp[:, None]) * np.log(1 - p),
        axis=0,
    )  # shape: (n_quad,)

    # unnormalized log posterior; subtract max for numerical stability
    log_post = log_lik + log_prior
    log_post -= log_post.max()
    post = np.exp(log_post)
    post /= post.sum()

    eap_mean = float(np.sum(theta_grid * post))
    eap_var  = float(np.sum((theta_grid - eap_mean) ** 2 * post))
    return eap_mean, eap_var


def Apply_Positive_Constraint(model, min_value=0.0):
    for param in model.parameters():
        param.data = torch.clamp(param.data, min=min_value)

In [ ]:
class Net(nn.Module):
    def __init__(self, input_size, first_hidden, second_hidden, action_space, dropout_rate):
        super(Net, self).__init__()
        self.fc1     = nn.Linear(input_size, first_hidden)
        self.fc2     = nn.Linear(first_hidden, second_hidden)
        self.out     = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(self.fc1(x))
        x = F.relu(x)
        x = self.dropout(self.fc2(x))
        x = F.relu(x)
        return self.out(x)

    def initialize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)

In [ ]:
def Choose_Action(item_id, state, epsilon):
    """state: 1-D array of shape (input_size,) = [theta_hat, log(post_var)]"""
    if np.random.rand() >= epsilon:
        state_t   = torch.unsqueeze(torch.FloatTensor(state), 0).to(device)
        item_id_t = torch.from_numpy(item_id).to(device).long()
        action_value = eval_net(state_t)
        action_value[:, item_id_t] = torch.zeros(item_id_t.shape).to(device)
        action = torch.max(action_value, -1)[1].cpu().numpy()
    else:
        if any(item_id):
            action = np.random.choice(np.delete(np.arange(action_space), item_id)).astype("int64").reshape(1,)
        else:
            action = np.random.choice(np.arange(action_space)).astype("int64").reshape(1,)
    return action


def Choose_Action_Test(item_id, state):
    """state: 2-D array of shape (input_size, n_subjects) = [[theta_hats], [log(post_vars)]]"""
    state_t      = torch.FloatTensor(state.swapaxes(0, 1)).to(device)
    action_value = eval_net(state_t).detach().cpu().numpy()
    if item_id.shape[0] > 0:
        action_value[
            np.tile(np.arange(item_id.shape[1])[np.newaxis, :], (item_id.shape[0], 1)),
            item_id,
        ] = np.zeros(item_id.shape)
    return action_value.argmax(axis=1)

In [ ]:
def TRAIN(cfg):
    best_valid = None
    best_state = None

    loss_func = nn.MSELoss()
    eval_net.train()
    optimizer = optim.Adam(eval_net.parameters(), lr=cfg.learning_rate)

    # memory layout: [state(2), action(1), reward(1), next_state(2)] = 6 cols
    memory         = np.zeros((cfg.memory_capacity, cfg.input_size * 2 + 2))
    memory_counter = 0
    learn_step_counter = 0

    if cfg.prior == "normal":
        training_theta = np.random.randn(cfg.training_size)
    elif cfg.prior == "uniform":
        training_theta = np.random.uniform(-3, 3, cfg.training_size)
    else:
        raise ValueError(f"Unsupported prior: {cfg.prior!r}. Use 'normal' or 'uniform'.")

    # log(prior_var): prior_std=1.0 -> log(1.0)=0.0
    log_prior_var = np.log(cfg.prior_std ** 2)

    for j in range(cfg.training_size):
        # Initial state: [theta_init ~ Uniform(-0.5,0.5), log(prior_variance)]
        state   = np.array([np.random.rand() - 0.5, log_prior_var])
        item_id = np.array([]).astype("int64")
        resp    = np.array([]).astype("int64")

        for i in range(cfg.test_length):
            action  = Choose_Action(item_id, state, cfg.epsilon)
            item_id = np.concatenate((item_id, action))
            resp    = np.concatenate((resp, RESPOND(item_bank[action], training_theta[j])))
            reward  = FI(item_bank[action,], training_theta[j])

            # EAP always converges regardless of all-correct / all-incorrect runs
            eap_mean, eap_var = EAP_quadrature(
                item_bank[item_id],
                resp,
                n_quad=cfg.n_quad,
                prior_mean=cfg.prior_mean,
                prior_std=cfg.prior_std,
            )
            next_state = np.array([eap_mean, np.log(max(eap_var, 1e-10))])

            memory[memory_counter % cfg.memory_capacity, :] = np.hstack(
                (state, action, reward, next_state)
            )
            memory_counter += 1
            state = next_state

            if memory_counter >= cfg.batch_size:
                batch_memory     = memory[np.random.choice(min(memory_counter, cfg.memory_capacity), cfg.batch_size), :]
                batch_state      = torch.FloatTensor(batch_memory[:, :cfg.input_size]).to(device)
                batch_action     = torch.LongTensor(batch_memory[:, cfg.input_size:cfg.input_size + 1].astype(int)).to(device)
                batch_reward     = torch.FloatTensor(batch_memory[:, cfg.input_size + 1:cfg.input_size + 2]).to(device)
                batch_next_state = torch.FloatTensor(batch_memory[:, -cfg.input_size:]).to(device)

                q_eval   = eval_net(batch_state).gather(1, batch_action)
                q_next   = target_net(batch_next_state).detach()
                q_target = (
                    batch_reward
                    if i == cfg.test_length - 1
                    else batch_reward + cfg.gamma * q_next.max(1)[0].view(cfg.batch_size, 1)
                )
                loss = loss_func(q_eval, q_target)

                optimizer.zero_grad()
                loss.backward()
                Apply_Positive_Constraint(eval_net)
                optimizer.step()

                learn_step_counter += 1
                if learn_step_counter % cfg.q_network_iteration == 0:
                    target_net.load_state_dict(eval_net.state_dict())

        ### Validation ###
        if (j + 1) % cfg.validation_interval == 0:
            eval_net.eval()
            valid_theta = np.random.choice(training_theta, cfg.validation_size)
            valid_bias  = np.zeros((cfg.test_length, cfg.validation_size))

            # state shape: (input_size=2, validation_size)
            # row 0: theta_hat,  row 1: log(posterior_variance)
            state = np.vstack([
                np.random.rand(cfg.validation_size) - 0.5,
                np.full(cfg.validation_size, log_prior_var),
            ])
            item_id = np.array([])

            for i in range(cfg.test_length):
                action = Choose_Action_Test(item_id, state)
                if i == 0:
                    item_id = action[np.newaxis, :]
                    resp    = RESPOND(item_bank[action,], valid_theta)[np.newaxis, :]
                else:
                    item_id = np.concatenate((item_id, action[np.newaxis, :]))
                    resp    = np.concatenate((resp, RESPOND(item_bank[action,], valid_theta)[np.newaxis, :]))

                theta_0 = np.zeros(cfg.validation_size)
                var_0   = np.zeros(cfg.validation_size)
                for s in range(cfg.validation_size):
                    theta_0[s], var_0[s] = EAP_quadrature(
                        item_bank[item_id[:, s]],
                        resp[:, s],
                        n_quad=cfg.n_quad,
                        prior_mean=cfg.prior_mean,
                        prior_std=cfg.prior_std,
                    )

                state = np.vstack([theta_0, np.log(np.maximum(var_0, 1e-10))])
                valid_bias[i] = theta_0 - valid_theta

            step_valid = np.transpose(np.vstack((
                np.arange(1, cfg.test_length + 1),
                np.mean(valid_bias, axis=1),
                np.sqrt(np.mean(valid_bias ** 2, axis=1)),
                np.mean(abs(valid_bias), axis=1),
            )))
            print("subject: {}\n\n{}\n".format(j + 1, step_valid))

            result_valid = np.mean(step_valid[6:, 1:], axis=0)
            if best_valid is None:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())
            elif result_valid[1] < best_valid[1]:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())

            eval_net.train()

    return best_state

In [ ]:
def TEST(cfg, theta_test):
    with torch.no_grad():
        eval_net.eval()
        testing_size = len(theta_test)

        log_prior_var = np.log(cfg.prior_std ** 2)

        # state shape: (input_size=2, testing_size)
        state = np.vstack([
            np.random.rand(testing_size) - 0.5,
            np.full(testing_size, log_prior_var),
        ])
        item_id  = np.array([])
        dqn_step = np.zeros((1, 4))

        for i in range(cfg.test_length):
            action = Choose_Action_Test(item_id, state)
            if i == 0:
                item_id = action[np.newaxis, :]
                resp    = RESPOND(item_bank[action,], theta_test)[np.newaxis, :]
            else:
                item_id = np.concatenate((item_id, action[np.newaxis, :]))
                resp    = np.concatenate((resp, RESPOND(item_bank[action,], theta_test)[np.newaxis, :]))

            theta_0 = np.zeros(testing_size)
            var_0   = np.zeros(testing_size)
            for s in range(testing_size):
                theta_0[s], var_0[s] = EAP_quadrature(
                    item_bank[item_id[:, s]],
                    resp[:, s],
                    n_quad=cfg.n_quad,
                    prior_mean=cfg.prior_mean,
                    prior_std=cfg.prior_std,
                )

            if i == 0:
                theta = theta_0[np.newaxis, :]
            else:
                theta = np.concatenate((theta, theta_0[np.newaxis, :]))

            dqn_step = np.vstack([dqn_step, np.array([
                i + 1,
                np.mean(theta_0 - theta_test),
                np.sqrt(np.mean((theta_0 - theta_test) ** 2)),
                np.mean(abs(theta_0 - theta_test)),
            ])])
            print("step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}".format(*dqn_step[-1]))

            state = np.vstack([theta_0, np.log(np.maximum(var_0, 1e-10))])

        user_id_col   = np.repeat(np.arange(1, testing_size + 1), cfg.test_length).reshape(-1, 1)
        step_col      = np.tile(np.arange(1, cfg.test_length + 1), testing_size).reshape(-1, 1)
        item_id_col   = (item_id + 1).transpose().reshape(-1, 1)
        resp_col      = resp.transpose().reshape(-1, 1)
        theta_est_col = theta.transpose().reshape(-1, 1)
        bias_col      = (theta - theta_test).transpose().reshape(-1, 1)

        dqn_data = pd.DataFrame(
            np.hstack([user_id_col, step_col, item_id_col, resp_col, theta_est_col, bias_col]),
            columns=["userID", "step", "itemID", "resp", "theta_est", "bias"],
        )
        dqn_summary = pd.DataFrame(dqn_step[1:], columns=["step", "Bias", "RMSE", "MAE"])

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        stem = f"{cfg.bank_type}_{cfg.bank_id}_DQN_{cfg.prior}_gamma_{cfg.gamma}"
        dqn_data.to_csv(   RESULTS_DIR / f"records_{stem}.csv",  index=False)
        dqn_summary.to_csv(RESULTS_DIR / f"summary_{stem}.csv",  index=False)
        print(f"\nSaved to {RESULTS_DIR}")

In [ ]:
cfg = Config(
    # Network
    input_size    = 2,      # [theta_hat, log(posterior_variance)]
    first_hidden  = 100,
    second_hidden = 50,
    dropout_rate  = 0.0,

    # Training
    test_length         = 40,
    gamma               = 0.1,
    memory_capacity     = 1000,
    epsilon             = 0.1,
    batch_size          = 128,
    q_network_iteration = 40,
    learning_rate       = 1e-3,
    training_size       = 1000,
    validation_size     = 200,
    validation_interval = 50,

    # EAP quadrature
    n_quad      = 61,
    prior_mean  = 0.0,
    prior_std   = 1.0,

    # Bank / prior
    bank_type = "uncor",
    bank_id   = 1,
    prior     = "normal",
    n_items   = 500,
)

bank_dir = {
    "uncor": ROOT / "data" / "uncorrelated_banks",
    "cor":   ROOT / "data" / "correlated_banks",
}[cfg.bank_type]

item_bank    = np.array(pd.read_csv(bank_dir / f"item_bank_{cfg.bank_type}_{cfg.bank_id}.csv")[["a", "b", "c"]])[:cfg.n_items]
action_space = item_bank.shape[0]

theta_test = np.array(pd.read_csv(ROOT / "data" / "theta_true" / f"theta_true_{cfg.bank_id}.csv")["x"])

print(f"item bank  : {item_bank.shape}")
print(f"theta_test : {theta_test.shape}")
print(f"\nConfig:\n{cfg}")

In [ ]:
eval_net   = Net(cfg.input_size, cfg.first_hidden, cfg.second_hidden, action_space, cfg.dropout_rate).to(device)
target_net = Net(cfg.input_size, cfg.first_hidden, cfg.second_hidden, action_space, cfg.dropout_rate).to(device)
eval_net.initialize()
target_net.initialize()

best_state = TRAIN(cfg)

assert best_state is not None, "No checkpoint was saved. Increase training_size or lower validation_interval."
eval_net.load_state_dict(best_state)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / f"dqn_{cfg.prior}_{cfg.bank_type}_{cfg.bank_id}_gamma_{cfg.gamma}.pt"
torch.save(eval_net.state_dict(), model_path)
print(f"Model saved to: {model_path}")

TEST(cfg, theta_test)